In [1]:
import pandas as pd
import numpy as np

# Statistial Analysis

Topics:

 - Normalization (Min-Max)
 - Mean, Median, Mode
 - Standard Deviation + ±3σ rule (outlier idea)
 - Standardization (Z-score)
 - IQR (Interquartile Range) outliers
 - Skewness, Kurtosis

In [2]:
# We'll create a feature vector with a few outliers so we can see:
# - mean vs median difference
# - ±3σ and IQR outlier detection
# - skewness/kurtosis behavior

# Create a feature vector by joining two arrays
x = np.concatenate([
    # Generate 95 random values (avg 50, std dev 10)
    np.random.normal(loc=50, scale=10, size=95),
    # Manually add 5 extreme high and low values
    np.array([120, 130, 140, 5, 8])
])

# Show the total number of elements in the array
print("Count:", x.size)
# Display only the first 10 values to check data
print("First 10:", x[:10])


Count: 100
First 10: [45.16810567 50.02222728 45.72416199 34.55542743 49.19217618 43.72769521
 52.00237844 36.26246798 53.89976944 51.10331498]


In [3]:
from IPython.display import HTML

HTML("""
<iframe width="560" height="315" src="https://www.youtube.com/embed/i373-Vc2d4o?si=g33RiXfjTcZHKD5d" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>
""")

In [4]:
# Mean   = average; sensitive to outliers, so it doesn't give accurate analysis
# Median = middle value after sorting; robust to outliers
# Mode   = most frequent value; useful for categorical/discrete data

# Calculate the average value of all elements
mean_x = np.mean(x)
# Find the middle value in the sorted dataset
median_x = np.median(x)

# Get unique values and their frequency counts
vals, counts = np.unique(x, return_counts=True)
# Find the value that appears most often
mode_x = vals[np.argmax(counts)]

# Display the calculated average
print("Mean  :", mean_x)
# Display the middle value
print("Median:", median_x)
# Display the most frequent value with a disclaimer
print("Mode  :", mode_x, "(Note: mode is not very meaningful for continuous random data)")


Mean  : 51.13057695427626
Median: 48.55511831190218
Mode  : 5.0 (Note: mode is not very meaningful for continuous random data)


In [5]:
# Std dev (σ) measures typical spread around the mean.
# ±3σ rule (empirical rule): for roughly normal data, ~99.7% points lie within mean ± 3σ.
# So values outside mean ± 3σ are often treated as potential outliers.

# Calculate standard deviation to measure data spread
std_x = np.std(x, ddof=0)

# Define the lower boundary (3 standard deviations below mean)
lower_3s = mean_x - 3 * std_x
# Define the upper boundary (3 standard deviations above mean)
upper_3s = mean_x + 3 * std_x

# Filter for values that fall outside these boundaries
outliers_3s = x[(x < lower_3s) | (x > upper_3s)]

# Display the standard deviation value
print("Std dev:", std_x)
# Display the calculated normal range
print("3σ range:", (lower_3s, upper_3s))
# List the specific values identified as outliers
print("Outliers by ±3σ rule:", outliers_3s)
# Show how many outliers were detected
print("Outlier count:", outliers_3s.size)


Std dev: 17.943690932478724
3σ range: (np.float64(-2.7004958431599135), np.float64(104.96164975171243))
Outliers by ±3σ rule: [120. 130. 140.]
Outlier count: 3


## Min–Max Normalization

$x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$

## Standardization (Z-score)

$z = \frac{x - \mu}{\sigma}$

where $\mu$ is the mean and $\sigma$ is the standard deviation.

## Cons of Min–Max and how Standardization helps

**Cons of Min–Max**

Very sensitive to outliers: if one value is extremely large/small, $x_{\max}$ or $x_{\min}$ becomes extreme, and most normal values get squeezed into a tiny range near 0 or 1.

Range depends on the dataset: when new data arrives with a new max/min, the scaling changes, so the same value can map differently across datasets.

**How Standardization helps**

Less dominated by extreme min/max: it scales using $\mu$ and $\sigma$, so a single outlier usually affects scaling less than directly setting the entire range.

Produces comparable feature scales: features become centered at 0 with unit variance, which often helps gradient-based models (e.g., logistic regression, SVM, neural nets) learn faster and more stably.

In [6]:
# Standardization: makes data have mean ~0 and std ~1
# Formula: z = (x - mean) / std
# Helpful for models sensitive to scale (e.g., gradient descent, SVM, KNN, PCA).

# Transform the data using the Z-score formula
x_z = (x - mean_x) / (std_x + 1e-8)

# Verify the new average is approximately zero
print("Standardized mean ~", x_z.mean())
# Verify the new spread is approximately one
print("Standardized std  ~", x_z.std())
# Preview the first five scaled values
print("First 5 standardized:", x_z[:5])


Standardized mean ~ 3.5971225997855074e-16
Standardized std  ~ 0.9999999994427012
First 5 standardized: [-0.33228789 -0.06176821 -0.30129893 -0.92373133 -0.10802687]


In [7]:
# Quartiles:
# Q1 = 25th percentile
# Q2 = 50th percentile (median)
# Q3 = 75th percentile
#
# IQR = Q3 - Q1 (spread of middle 50% of data)
#-----------q1--------/--------q2-------/---------q3-------/----------q4--------/
# IQR outlier rule (common robust rule):
# lower = Q1 - 1.5*IQR
# upper = Q3 + 1.5*IQR
# Values outside are potential outliers (more robust than ±3σ for skewed data).

# Calculate the value at the 25th percentile
Q1 = np.percentile(x, 25)
# Calculate the value at the 75th percentile
Q3 = np.percentile(x, 75)
# Find the range containing the middle 50% of data
IQR = Q3 - Q1

# Define the lower outlier limit using 1.5x IQR
lower_iqr = Q1 - 1.5 * IQR
# Define the upper outlier limit using 1.5x IQR
upper_iqr = Q3 + 1.5 * IQR

# Filter the array for values outside the IQR boundaries
outliers_iqr = x[(x < lower_iqr) | (x > upper_iqr)]

# Display the three quartile points
print("Q1, Median, Q3:", Q1, median_x, Q3)
# Show the calculated Interquartile Range
print("IQR:", IQR)
# Display the boundaries used for outlier detection
print("IQR range:", (lower_iqr, upper_iqr))
# List the values flagged as outliers by this method
print("Outliers by IQR:", outliers_iqr)
# Show the total number of outliers detected
print("Outlier count:", outliers_iqr.size)


Q1, Median, Q3: 43.24422725759184 48.55511831190218 56.6063141540863
IQR: 13.36208689649446
IQR range: (np.float64(23.201096912850147), np.float64(76.64944449882799))
Outliers by IQR: [ 78.04594871  78.60356169 120.         130.         140.
   5.           8.        ]
Outlier count: 7


In [10]:
# When to use what (quick rules):
# - Mean/Std: good for roughly normal numeric features (but sensitive to outliers)
# - Median/IQR: robust summaries when outliers/skew exist
# - Min-Max normalization: when you want [0,1] scaling (but outlier-sensitive)
# - Z-score standardization: common default for many ML algorithms
# - ±3σ rule: simple outlier rule for near-normal data
# - IQR rule: robust outlier rule for skewed/non-normal data

# Create a dictionary to store all calculated statistics
summary = {
    "mean": mean_x,
    "median": median_x,
    "std": std_x,
    # Calculate the minimum value in the array
    "min": np.min(x),
    # Calculate the maximum value in the array
    "max": np.max(x),
    "Q1": Q1,
    "Q3": Q3,
    "IQR": IQR,
    # Convert outlier count from 3-sigma method to integer
    "outliers_3sigma_count": int(outliers_3s.size),
    # Convert outlier count from IQR method to integer
    "outliers_iqr_count": int(outliers_iqr.size),
}
# Convert the dictionary into a Pandas Series for clean display
pd.Series(summary)



,0
mean,51.130577
median,48.555118
std,17.943691
min,5.000000
max,140.000000
Q1,43.244227
Q3,56.606314
IQR,13.362087
outliers_3sigma_count,3.000000
outliers_iqr_count,7.000000
